# v34 ランカー lab — 「576本のどれをどれだけ信用するか」の手法比較(1回=1変更)
基準: コンセンサス(等重みmedian) / **初代LGBMランカー 7.9452** / oracle(1本) 4.147。

**固定プロトコル(全プリセット共通)**: fold割り共有(`ranker_folds.json`) / q̂は井戸内z標準化 →
softmax(−β·q̂z), β∈{0.25,0.5,1,2} + top-n∈{4,16,64} の粗グリッドのみ / 報告=overall・oracleギャップ回収率・
井戸別勝率・硬い核・fold間分散 / 結果は `ranker_lab_results.json` に蓄積。

**ガードレール(過去の轍)**: ラベル=連続値の井戸内z(argmax禁止=v20) / blend・GBM・NN予測由来の特徴禁止
(参照=コンセンサス=v24対策) / oracleは表示のみで判定に使わない / 採用=現ベスト比−0.05以上 /
`ctrl_shuffle`(井戸内ラベル並べ替え)で回収≈0を確認=リーク検出。

| RUN | 内容 |
|---|---|
| base | LGBM回帰・スカラ27特徴(初代の再現、ただしq̂z標準化プロトコル) |
| b_rank | 同特徴で LGBMRanker(lambdarank, 8段階relevance) |
| ctrl_shuffle | base のラベルを井戸内シャッフル(対照。回収が出たらリーク) |
| c_tcn | 系列TCN: オフセット/エンベロープ/GR残差×3/dip/GR文脈 の7ch → listwise KL |
| d_meta | c_tcn + メタデータ(src/param/sm-fw/ww/モード文脈)を埋め込みに連結 |
| e_attn | d_meta + 井戸内 self-attention 1層(候補集合の文脈=set-encoder) |


In [ ]:
# ===== 共通ロード + 固定fold =====
import json, warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
warnings.filterwarnings("ignore")
matplotlib.rcParams["font.family"] = "Meiryo"; matplotlib.rcParams["axes.unicode_minus"] = False

PROJECT = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
V28_ART = PROJECT / "v28" / "artifacts_v28"
V32_ART = PROJECT / "v32" / "artifacts_v32"
ART = PROJECT / "v34" / "artifacts_v34"

tbl = pq.read_table(V28_ART / "data" / "train.parquet",
                    columns=["well", "target", "last_known_tvt", "md_since",
                             "pf_nbr_has", "pf_nbr_nrefs", "pf_nbr_nndist"])
g = tbl.column("well").to_pandas().to_numpy()
y0 = tbl.column("target").to_numpy(zero_copy_only=False).astype(np.float64)
lk = tbl.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
mds = tbl.column("md_since").to_numpy(zero_copy_only=False).astype(np.float64)
has = tbl.column("pf_nbr_has").to_numpy(zero_copy_only=False).astype(np.float64)
nrefs_c = tbl.column("pf_nbr_nrefs").to_numpy(zero_copy_only=False).astype(np.float64)
nnd_c = tbl.column("pf_nbr_nndist").to_numpy(zero_copy_only=False).astype(np.float64)
tvt_true = lk + y0
wid = list(pd.unique(g))
rows_of = {w: np.where(g == w)[0] for w in wid}
HARD = set(pd.read_csv(V32_ART / "hardcore_diag.csv", index_col=0).query(
    "rmse_prior >= 10 and abs(bias) > 8").index) & set(wid)
DUMPS = {k: np.load(ART / f"seed_dump_pf{k}.npz") for k in (1, 2, 3)}
KEYS = {k: set(DUMPS[k].files) for k in (1, 2, 3)}
def rmse(e):
    e = e[np.isfinite(e)]
    return float(np.sqrt(np.mean(e ** 2))) if e.size else np.nan

# 固定fold(全プリセット共有。無ければ作って保存)
FOLD_FP = ART / "ranker_folds.json"
if FOLD_FP.exists():
    FOLD = {k: int(v) for k, v in json.loads(FOLD_FP.read_text()).items()}
else:
    from sklearn.model_selection import GroupKFold
    FOLD = {}
    for f, (_, va) in enumerate(GroupKFold(5).split(np.zeros(len(wid)), groups=np.array(wid))):
        for i in va: FOLD[wid[i]] = f
    FOLD_FP.write_text(json.dumps(FOLD))
print(f"wells {len(wid)} | 硬い核 {len(HARD)} | folds 固定({FOLD_FP.name})")


In [ ]:
# ===== 軌跡ストア(TRAJ) + スカラ特徴テーブル(TT)の構築(全プリセット共通の土台) =====
GAP_LINK = 8.0
SRCS = ("tw", "self", "nbr"); FLTS = (("sm", "ps"), ("fw", "pfw"))
SRC_ID = {"tw": 0, "self": 1, "nbr": 2}

def score_batch(TR, gg, gmin, gst, gs, gr):
    G = len(gg)
    ii = np.clip((TR - gmin) / gst, 0.0, G - 1.000001)
    i0 = ii.astype(int); t = ii - i0
    eg = gg[i0] * (1 - t) + gg[np.minimum(i0 + 1, G - 1)] * t
    m = np.isfinite(gr)[None, :] & np.isfinite(eg)
    d = np.where(m, (gr[None, :] - eg) / gs, np.nan)
    with np.errstate(all="ignore"):
        return np.nanmean(np.exp(-0.5 * d * d), axis=1), np.where(m, gr[None, :] - eg, np.nan)

def cluster(ds, wts):
    o = np.argsort(ds)
    groups, cur = [], [o[0]]
    for i in o[1:]:
        if ds[i] - ds[cur[-1]] > GAP_LINK: groups.append(cur); cur = [i]
        else: cur.append(i)
    groups.append(cur)
    lab = np.empty(len(ds), int); minfo = []
    for mi, m in enumerate(groups):
        m = np.array(m); lab[m] = mi
        minfo.append(dict(mass=float(wts[m].sum()), idx=m))
    return lab, minfo

t0 = time.time(); rows = []; TRAJ = {}
for iw, w in enumerate(wid):
    r = rows_of[w]
    nbr_ok = np.nanmean(has[r]) > 0.5
    trajs, wts, meta_run = [], [], []
    for k in (1, 2, 3):
        z = DUMPS[k]
        for src in SRCS:
            if src == "nbr" and not nbr_ok: continue
            if f"{w}|ps_{src}" not in KEYS[k]: continue
            ww_ = z[f"{w}|ww_{src}"]; ll_ = z[f"{w}|ll_{src}"]
            for flt, pref in FLTS:
                trajs.append(z[f"{w}|{pref}_{src}"]); wts.append(ww_)
                meta_run.append((k, src, flt, ll_))
    if not trajs: continue
    n_runs = len(trajs)
    T = min(min(t_.shape[1] for t_ in trajs), len(r))
    if T < 5: continue
    ALL = np.concatenate([t_[:, :T].astype(np.float64) for t_ in trajs], 0)
    IW = np.concatenate([wv / n_runs for wv in wts])
    cons = np.median(ALL, axis=0)
    p10 = np.percentile(ALL, 10, axis=0); p90 = np.percentile(ALL, 90, axis=0)
    ds = np.nanmedian(ALL - cons[None, :], axis=1)
    fin = np.isfinite(ds)
    if fin.sum() < 2: continue
    lab, minfo = cluster(np.where(fin, ds, 0.0), IW)
    tot_mass = max(IW.sum(), 1e-12)
    top_mi = int(np.argmax([m["mass"] for m in minfo]))
    run_src = np.concatenate([[src] * t_.shape[0] for (k, src, flt, ll_), t_ in zip(meta_run, trajs)])
    ext_mass_of = {mi: float(IW[(lab == mi) & np.isin(run_src, ("self", "nbr"))].sum())
                   for mi in range(len(minfo))}
    z1 = DUMPS[1]
    md_ = z1[f"{w}|md"].astype(np.float64)[:T]; zz = z1[f"{w}|z"].astype(np.float64)[:T]
    ir = float(z1[f"{w}|ir"])
    dmd = np.maximum(np.diff(md_), 1.0)
    GT = np.diff(ALL + zz[None, :], axis=1) / dmd[None, :]
    dip_med = np.nanmedian(GT, axis=1); dip_mx = np.nanpercentile(np.abs(GT), 95, axis=1)
    jump = np.nanmax(np.abs(np.diff(ALL + zz[None, :], axis=1)), axis=1)
    r2 = np.nanmean(np.abs(np.diff(ALL, 2, axis=1)), axis=1)
    b2 = float(np.nanmean(np.abs(np.diff(cons, 2)))) + 1e-6
    xsc = {kk: np.full(len(ALL), np.nan) for kk in ("tw", "self", "nbr")}
    pos = 0
    for (k, src, flt, ll_), t_ in zip(meta_run, trajs):
        S = t_.shape[0]; sl = slice(pos, pos + S); pos += S
        z = DUMPS[k]
        gmin, gst, gs = z[f"{w}|meta"]; gr = z[f"{w}|gr"].astype(np.float64)[:T]
        for nmk, gk in [("tw", "gg_tw"), ("self", "gg_self"), ("nbr", "gg_nbr")]:
            if f"{w}|{gk}" not in KEYS[k]: continue
            gg = z[f"{w}|{gk}"].astype(np.float64)
            sc, _ = score_batch(ALL[sl], gg, gmin, gst, gs, gr)
            s0, _ = score_batch(cons[None, :], gg, gmin, gst, gs, gr)
            xsc[nmk][sl] = sc - s0[0]
    smfw = np.full(len(ALL), np.nan)
    offs = np.cumsum([0] + [t_.shape[0] for t_ in trajs])
    for i_run, (k, src, flt, ll_) in enumerate(meta_run):
        j_run = i_run + 1 if flt == "sm" else i_run - 1
        if 0 <= j_run < n_runs and meta_run[j_run][1] == src and meta_run[j_run][0] == k:
            a = ALL[offs[i_run]:offs[i_run + 1]]; b = ALL[offs[j_run]:offs[j_run + 1]]
            smfw[offs[i_run]:offs[i_run + 1]] = np.nanmean(np.abs(a - b), axis=1)
    err = np.nanmedian(np.abs(ALL - tvt_true[r[:T]][None, :]), axis=1)
    ez = (err - np.nanmean(err)) / (np.nanstd(err) + 1e-9)
    md_len = float(np.nanmax(mds[r])) if np.isfinite(mds[r]).any() else np.nan
    item_k = np.concatenate([[k] * t_.shape[0] for (k, src, flt, ll_), t_ in zip(meta_run, trajs)])
    item_s = np.concatenate([[SRC_ID[src]] * t_.shape[0] for (k, src, flt, ll_), t_ in zip(meta_run, trajs)])
    item_f = np.concatenate([[int(flt == "sm")] * t_.shape[0] for (k, src, flt, ll_), t_ in zip(meta_run, trajs)])
    pos = 0
    for i_run, ((k, src, flt, ll_), wv, t_) in enumerate(zip(meta_run, wts, trajs)):
        S = t_.shape[0]
        llz = (ll_ - np.nanmean(ll_)) / (np.nanstd(ll_) + 1e-9)
        wrank = np.argsort(np.argsort(-wv)) / max(S - 1, 1)
        for s in range(S):
            i = pos + s
            if not fin[i]: continue
            rows.append(dict(
                well=w, fold=FOLD[w], err=float(err[i]), y=float(ez[i]),
                src_tw=int(src == "tw"), src_self=int(src == "self"), src_nbr=int(src == "nbr"),
                pset=k, flt_sm=int(flt == "sm"),
                ww=float(wv[s]), ww_rank=float(wrank[s]), ll_z=float(llz[s]),
                d=float(ds[i]), d_abs=float(abs(ds[i])),
                mode_mass=float(minfo[lab[i]]["mass"] / tot_mass),
                mode_is_top=int(lab[i] == top_mi),
                mode_ext_mass=float(ext_mass_of[lab[i]] / tot_mass),
                x_tw=float(xsc["tw"][i]), x_self=float(xsc["self"][i]), x_nbr=float(xsc["nbr"][i]),
                dip_dev=float(abs(dip_med[i] - ir)), dip_absmax=float(dip_mx[i]),
                geo_jump=float(jump[i]), rough_ratio=float(r2[i] / b2),
                smfw_gap=float(smfw[i]),
                seed_disp=float(np.nanstd(ds[fin])),
                nbr_ok=int(nbr_ok), nndist=float(np.nanmedian(nnd_c[r])),
                nrefs=float(np.nanmedian(nrefs_c[r])), n_rows=float(len(r)), md_len=md_len))
        pos += S
    TRAJ[w] = dict(ALL=ALL[fin].astype(np.float32), T=T, err=err[fin].astype(np.float32),
                   cons=cons.astype(np.float32), p10=p10.astype(np.float32), p90=p90.astype(np.float32),
                   k=item_k[fin].astype(np.int8), srci=item_s[fin].astype(np.int8),
                   flt=item_f[fin].astype(np.int8))
    if (iw + 1) % 150 == 0:
        print(f"  {iw+1}/{len(wid)} wells ({time.time()-t0:.0f}s)")
TT = pd.DataFrame(rows)
for w in TRAJ:
    assert (TT["well"] == w).sum() == len(TRAJ[w]["ALL"])   # 行整列の保証
cons_ov = rmse(np.concatenate([TRAJ[w]["cons"][:TRAJ[w]["T"]] - tvt_true[rows_of[w][:TRAJ[w]["T"]]] for w in TRAJ]))
or1_ov = rmse(np.concatenate([TRAJ[w]["ALL"][int(np.argmin(TRAJ[w]["err"]))] - tvt_true[rows_of[w][:TRAJ[w]["T"]]] for w in TRAJ]))
TT_IDX = {w: sub.index.to_numpy() for w, sub in TT.groupby("well", sort=False)}

def quick_eval(qvals, wells, beta=0.5):
    """fold途中経過用の即席評価: softmax β0.5 集約 vs 同じ井戸のコンセンサス。"""
    e_e, e_c = [], []
    for w in wells:
        q = qvals[TT_IDX[w]].astype(np.float64)
        q = (q - q.mean()) / (q.std() + 1e-9)
        wgt = np.exp(-beta * q); wgt = wgt / wgt.sum()
        d = TRAJ[w]
        est = (wgt[:, None] * d["ALL"].astype(np.float64)).sum(0)
        tr_ = tvt_true[rows_of[w][:d["T"]]]
        e_e.append(est - tr_); e_c.append(d["cons"][:d["T"]].astype(np.float64) - tr_)
    return rmse(np.concatenate(e_e)), rmse(np.concatenate(e_c))

print(f"TT {len(TT)}行 / {TT['well'].nunique()}井戸 ({time.time()-t0:.0f}s)")
print(f"基準: コンセンサス {cons_ov:.4f} / oracle(1本) {or1_ov:.4f}(oracleは表示のみ・判定に使わない)")


In [ ]:
# ===== プリセット選択(1回=1変更) =====
PRESETS = {
    "base":         dict(kind="lgbm", rank=False, shuffle=False),
    "b_rank":       dict(kind="lgbm", rank=True,  shuffle=False),
    "ctrl_shuffle": dict(kind="lgbm", rank=False, shuffle=True),   # 対照: 回収が出たらリーク
    "c_tcn":        dict(kind="seq", meta=False, attn=False),
    "d_meta":       dict(kind="seq", meta=True,  attn=False),
    "e_attn":       dict(kind="seq", meta=True,  attn=True),
}
RUN = "base"
CFG = PRESETS[RUN]
print("RUN =", RUN, CFG)

FEATS = ["src_tw", "src_self", "src_nbr", "pset", "flt_sm", "ww", "ww_rank", "ll_z",
         "d", "d_abs", "mode_mass", "mode_is_top", "mode_ext_mass",
         "x_tw", "x_self", "x_nbr", "dip_dev", "dip_absmax", "geo_jump", "rough_ratio",
         "smfw_gap", "seed_disp", "nbr_ok", "nndist", "nrefs", "n_rows", "md_len"]
META_FEATS = ["src_tw", "src_self", "src_nbr", "pset", "flt_sm", "ww", "ww_rank", "ll_z",
              "d", "mode_mass", "mode_is_top", "mode_ext_mass"]


In [ ]:
# ===== スカラ系プリセット(base / b_rank / ctrl_shuffle) =====
if CFG["kind"] == "lgbm":
    import lightgbm as lgb
    rng = np.random.default_rng(0)
    ylab = TT["y"].to_numpy(np.float32).copy()
    if CFG["shuffle"]:
        for w, sub in TT.groupby("well"):
            idx = sub.index.to_numpy(); ylab[idx] = ylab[rng.permutation(idx)]
        print("★対照run: ラベルを井戸内シャッフル済(回収≈0が正常)")
    X = np.nan_to_num(TT[FEATS].to_numpy(np.float32), nan=-999.0)
    foldc = TT["fold"].to_numpy()
    q_oof = np.zeros(len(TT), np.float32)
    for f in range(5):
        tr = np.where(foldc != f)[0]; va = np.where(foldc == f)[0]
        fp = ART / f"lab_q_{RUN}_f{f}.npy"          # fold チェックポイント(設定を変えたら削除すること)
        if fp.exists():
            q_oof[va] = np.load(fp)
            vw = [w for w in TRAJ if FOLD[w] == f]
            ee, cc = quick_eval(q_oof, vw)
            print(f"fold{f} [cache]: 集約(β0.5) {ee:.4f} vs cons {cc:.4f}")
            continue
        if CFG["rank"]:
            sub = TT.iloc[tr]
            order = np.argsort(sub["well"].to_numpy(), kind="stable")
            tr_o = tr[order]
            wells_o = TT["well"].to_numpy()[tr_o]
            grp_sizes = pd.Series(wells_o).groupby(wells_o, sort=False).size().to_numpy()
            errv = TT["err"].to_numpy()[tr_o]
            rel = np.empty(len(tr_o), int); s0 = 0
            for gs_ in grp_sizes:
                e = errv[s0:s0 + gs_]
                pct = np.argsort(np.argsort(e)) / max(gs_ - 1, 1)
                rel[s0:s0 + gs_] = np.clip((7 * (1 - pct)).astype(int), 0, 7)
                s0 += gs_
            mdl = lgb.LGBMRanker(objective="lambdarank", n_estimators=700, learning_rate=0.05,
                                 num_leaves=63, min_child_samples=100, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.8, reg_lambda=1.0, verbose=-1,
                                 label_gain=[2 ** i - 1 for i in range(8)])
            mdl.fit(X[tr_o], rel, group=grp_sizes)
            q_oof[va] = -mdl.predict(X[va])          # score大=良 → q=悪さ に反転
        else:
            mdl = lgb.LGBMRegressor(n_estimators=700, learning_rate=0.05, num_leaves=63,
                                    min_child_samples=100, subsample=0.8, subsample_freq=1,
                                    colsample_bytree=0.8, reg_lambda=1.0, verbose=-1)
            mdl.fit(X[tr], ylab[tr])
            q_oof[va] = mdl.predict(X[va])
        np.save(fp, q_oof[va])
        vw = [w for w in TRAJ if FOLD[w] == f]
        ee, cc = quick_eval(q_oof, vw)
        print(f"fold{f} done: 集約(β0.5) {ee:.4f} vs cons {cc:.4f}(このfoldの井戸のみ)")
    TT[f"q_{RUN}"] = q_oof


In [ ]:
# ===== 系列系プリセット(c_tcn / d_meta / e_attn) =====
if CFG["kind"] == "seq":
    import torch, torch.nn as nn, torch.nn.functional as F
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    STRIDE, TAU, EPOCHS, LR = 8, 0.5, 12, 3e-4
    torch.manual_seed(0)

    def build_channels(w):
        """(n_traj, 7, Td) float16。★間引き後の系列で計算(前計算を約8倍高速化)。"""
        d = TRAJ[w]; T = d["T"]
        ALL = d["ALL"][:, ::STRIDE].astype(np.float32)
        cons = d["cons"][::STRIDE]; p10 = d["p10"][::STRIDE]; p90 = d["p90"][::STRIDE]
        n = len(ALL)
        z1 = DUMPS[1]
        md_ = z1[f"{w}|md"].astype(np.float32)[:T][::STRIDE]
        zz = z1[f"{w}|z"].astype(np.float32)[:T][::STRIDE]
        ir = float(z1[f"{w}|ir"])
        off = np.clip((ALL - cons[None, :]) / 10.0, -6, 6)
        env = np.clip((ALL - p10[None, :]) / (p90 - p10 + 1e-3)[None, :] - 0.5, -3, 3)
        dmd = np.maximum(np.diff(md_), 1.0)
        dip = np.zeros_like(ALL)
        dip[:, 1:] = np.clip((np.diff(ALL + zz[None, :], axis=1) / dmd[None, :] - ir) * 100.0, -4, 4)
        RES = {kk: np.zeros_like(ALL) for kk in ("tw", "self", "nbr")}
        for k in (1, 2, 3):
            m = d["k"] == k
            if not m.any(): continue
            z = DUMPS[k]
            gmin, gst, gs = z[f"{w}|meta"]; gr = z[f"{w}|gr"].astype(np.float32)[:T][::STRIDE]
            for nmk, gk in [("tw", "gg_tw"), ("self", "gg_self"), ("nbr", "gg_nbr")]:
                if f"{w}|{gk}" not in KEYS[k]: continue
                gg = z[f"{w}|{gk}"].astype(np.float32)
                G = len(gg)
                ii = np.clip((ALL[m] - gmin) / gst, 0.0, G - 1.000001)
                i0 = ii.astype(int); t = ii - i0
                eg = gg[i0] * (1 - t) + gg[np.minimum(i0 + 1, G - 1)] * t
                RES[nmk][m] = np.clip(np.nan_to_num((gr[None, :] - eg) / max(float(gs), 1e-3)), -4, 4)
        grz = z1[f"{w}|gr"].astype(np.float32)[:T][::STRIDE]
        grz = np.clip((grz - np.nanmedian(grz)) / (np.nanstd(grz) + 1e-3), -4, 4)
        grz = np.nan_to_num(np.broadcast_to(grz, (n, len(grz))))
        Xw = np.stack([off, env, RES["tw"], RES["self"], RES["nbr"], dip, grz], 1)
        return np.ascontiguousarray(Xw).astype(np.float16)

    class ChanLN(nn.Module):
        def __init__(self, c):
            super().__init__(); self.ln = nn.LayerNorm(c)
        def forward(self, x): return self.ln(x.transpose(1, 2)).transpose(1, 2)
    class Scorer(nn.Module):
        def __init__(self, cin=7, ch=32, dmeta=len(META_FEATS), use_meta=False, use_attn=False):
            super().__init__()
            self.use_meta, self.use_attn = use_meta, use_attn
            self.inp = nn.Conv1d(cin, ch, 1)
            self.blocks = nn.ModuleList()
            for dil in (1, 2, 4, 8):
                self.blocks.append(nn.ModuleList([
                    nn.Conv1d(ch, ch, 5, padding=2 * dil, dilation=dil), ChanLN(ch), nn.Conv1d(ch, ch, 1)]))
            self.emb = nn.Linear(2 * ch, 64)
            if use_meta: self.mp = nn.Sequential(nn.Linear(dmeta, 64), nn.GELU(), nn.Linear(64, 64))
            if use_attn:
                self.q = nn.Linear(64, 64); self.kk = nn.Linear(64, 64); self.v = nn.Linear(64, 64)
                self.aln = nn.LayerNorm(64)
            self.head = nn.Sequential(nn.Linear(64, 64), nn.GELU(), nn.Linear(64, 1))
        def forward(self, x, meta=None):
            h = self.inp(x)
            for cv, ln, pw in self.blocks:
                h = h + pw(F.gelu(ln(cv(h))))
            e = self.emb(torch.cat([h.mean(-1), h.amax(-1)], -1))
            if self.use_meta and meta is not None: e = e + self.mp(meta)
            if self.use_attn:
                a = F.scaled_dot_product_attention(self.q(e)[None], self.kk(e)[None], self.v(e)[None])[0]
                e = self.aln(e + a)
            return self.head(e).squeeze(-1)

    print("チャネル前計算(GPU常駐, fp16) ...", flush=True); t0 = time.time()
    CH = {w: torch.from_numpy(build_channels(w)).to(DEV) for w in TRAJ}   # fp16のままGPUへ
    MB = {w: torch.from_numpy(np.nan_to_num(sub[META_FEATS].to_numpy(np.float32), nan=0.0)).to(DEV)
          for w, sub in TT.groupby("well", sort=False)}
    PSTAR = {w: F.softmax(torch.from_numpy(-sub["y"].to_numpy(np.float32) / TAU).to(DEV), 0)
             for w, sub in TT.groupby("well", sort=False)}
    vram = sum(v.element_size() * v.nelement() for v in CH.values()) / 1e9
    HB = float(np.mean([-(p * torch.log(p.clamp_min(1e-12))).sum().item() for p in PSTAR.values()]))
    UNI = float(np.mean([np.log(len(p)) for p in PSTAR.values()]))
    print(f"  done {time.time()-t0:.0f}s(VRAM ~{vram:.1f}GB)")
    print(f"  lossの物差し: 無学習(一様)={UNI:.3f} / 理論下限 H(p*)={HB:.3f} → 学習可能幅 {UNI-HB:.3f}。"
          f"epoch表示の KL=loss−H(p*) が0に近いほど正解分布に一致")

    q_oof = np.zeros(len(TT), np.float32)
    wells_all = [w for w in TRAJ]
    for f in range(5):
        trw = [w for w in wells_all if FOLD[w] != f]; vaw = [w for w in wells_all if FOLD[w] == f]
        fp = ART / f"lab_q_{RUN}_f{f}.npy"          # fold チェックポイント(設定を変えたら削除すること)
        if fp.exists():
            va_idx = np.concatenate([TT_IDX[w] for w in vaw])
            q_oof[va_idx] = np.load(fp)
            ee, cc = quick_eval(q_oof, vaw)
            print(f"fold{f} [cache]: 集約(β0.5) {ee:.4f} vs cons {cc:.4f}", flush=True)
            continue
        mdl = Scorer(use_meta=CFG["meta"], use_attn=CFG["attn"]).to(DEV)
        opt = torch.optim.AdamW(mdl.parameters(), lr=LR, weight_decay=1e-4)
        rng = np.random.default_rng(f)
        for ep in range(EPOCHS):
            te = time.time(); mdl.train(); order = rng.permutation(len(trw)); tot = 0.0
            for j in order:
                w = trw[j]
                s = mdl(CH[w].float(), MB[w] if CFG["meta"] else None)
                loss = -(PSTAR[w] * F.log_softmax(s, 0)).sum()
                opt.zero_grad(); loss.backward(); opt.step(); tot += float(loss)
            if ep % 3 == 2:
                la = tot / len(trw)
                print(f"fold{f} ep{ep+1}: loss {la:.4f}(KL {la-HB:.4f} / 進捗 "
                      f"{100*(UNI-la)/max(UNI-HB,1e-9):.0f}% / {time.time()-te:.0f}s/ep)", flush=True)
        mdl.eval()
        with torch.no_grad():
            for w in vaw:
                q_oof[TT_IDX[w]] = -mdl(CH[w].float(), MB[w] if CFG["meta"] else None).float().cpu().numpy()
        va_idx = np.concatenate([TT_IDX[w] for w in vaw])
        np.save(fp, q_oof[va_idx])
        ee, cc = quick_eval(q_oof, vaw)
        print(f"fold{f} done: 集約(β0.5) {ee:.4f} vs cons {cc:.4f}(このfoldの井戸のみ)", flush=True)
    TT[f"q_{RUN}"] = q_oof


In [ ]:
# ===== 固定プロトコルで評価 → results 蓄積 =====
qcol = f"q_{RUN}"
Q = {}
for w, sub in TT.groupby("well", sort=False):
    q = sub[qcol].to_numpy(np.float64)
    Q[w] = (q - q.mean()) / (q.std() + 1e-9)          # 井戸内z標準化(プリセット間でβ共通)

def agg_overall(mode, param, wells=None):
    e_all = []
    for w in (wells or TRAJ):
        d = TRAJ[w]; q = Q[w]
        if mode == "softmax":
            wgt = np.exp(-param * q); wgt = wgt / wgt.sum()
        else:
            kk = min(param, len(q)); idx = np.argsort(q)[:kk]
            wgt = np.zeros(len(q)); wgt[idx] = 1.0 / kk
        est = (wgt[:, None] * d["ALL"].astype(np.float64)).sum(0)
        e_all.append(est - tvt_true[rows_of[w][:d["T"]]])
    return rmse(np.concatenate(e_all))

res = [(f"softmax β{b}", "softmax", b) for b in (0.25, 0.5, 1.0, 2.0)] + \
      [(f"top-{n}", "topn", n) for n in (4, 16, 64)]
scores = [(nm, agg_overall(mo, pa)) for nm, mo, pa in res]
best_nm, best_ov = min(scores, key=lambda x: x[1])
bmode, bparam = [(mo, pa) for nm, mo, pa in res if nm == best_nm][0]
recov = 100 * (cons_ov - best_ov) / max(cons_ov - or1_ov, 1e-9)
# 井戸別勝率 / 硬い核 / fold間分散(best則)
win = 0; hard_gain = 0.0; per_fold = []
for f in range(5):
    per_fold.append(agg_overall(bmode, bparam, wells=[w for w in TRAJ if FOLD[w] == f]))
for w in TRAJ:
    d = TRAJ[w]; q = Q[w]
    if bmode == "softmax":
        wgt = np.exp(-bparam * q); wgt = wgt / wgt.sum()
    else:
        kk = min(bparam, len(q)); idx = np.argsort(q)[:kk]
        wgt = np.zeros(len(q)); wgt[idx] = 1.0 / kk
    est = (wgt[:, None] * d["ALL"].astype(np.float64)).sum(0)
    tr_ = tvt_true[rows_of[w][:d["T"]]]
    ec = rmse(d["cons"][:d["T"]].astype(np.float64) - tr_); en = rmse(est - tr_)
    if en < ec: win += 1
    if w in HARD: hard_gain += en - ec

print(f"[{RUN}] " + " | ".join(f"{nm}:{ov:.4f}" for nm, ov in scores))
print(f"best {best_nm} = {best_ov:.4f}(cons {cons_ov:.4f} / oracle {or1_ov:.4f} → 回収 {recov:.1f}%)")
print(f"井戸別勝率 {win}/{len(TRAJ)} | 硬い核ΔRMSE合計 {hard_gain:+.1f} | fold別 {['%.3f' % v for v in per_fold]}")

RES_FP = ART / "ranker_lab_results.json"
allres = json.loads(RES_FP.read_text()) if RES_FP.exists() else {}
allres[RUN] = dict(best_rule=best_nm, overall=best_ov, recovery_pct=recov,
                   win=win, n_wells=len(TRAJ), hard_gain=hard_gain,
                   fold_overall=per_fold, grid={nm: ov for nm, ov in scores})
RES_FP.write_text(json.dumps(allres, indent=1))
prev = {k: v["overall"] for k, v in allres.items() if k != "ctrl_shuffle" and k != RUN}
if RUN == "ctrl_shuffle":
    print("→ 対照run: 回収≈0%(consと同等)なら健全。有意に回収していたらリーク疑い")
elif prev:
    bb = min(prev.values())
    print(f"→ 採用判定: 現ベスト {bb:.4f} に対し {best_ov - bb:+.4f}(採用ライン −0.05)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
bs = [(pa, ov) for (nm, mo, pa), (_, ov) in zip(res, scores) if mo == "softmax"]
ax[0].plot([b for b, _ in bs], [o for _, o in bs], "o-", label=RUN)
ax[0].axhline(cons_ov, c="k", ls="--", label="コンセンサス")
ax[0].axhline(7.9452, c="g", ls=":", label="初代LGBM(7.945)")
ax[0].set_xlabel("β"); ax[0].set_ylabel("overall"); ax[0].legend(fontsize=8); ax[0].set_title("softmax集約")
names = list(allres.keys()); vals = [allres[k]["overall"] for k in names]
ax[1].bar(names, vals, color=["gray" if n == "ctrl_shuffle" else "steelblue" for n in names])
ax[1].axhline(cons_ov, c="k", ls="--"); ax[1].set_title("プリセット比較(蓄積)")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()


In [ ]:
# ===== 蓄積結果の一覧 =====
allres = json.loads((ART / "ranker_lab_results.json").read_text())
df = pd.DataFrame(allres).T[["best_rule", "overall", "recovery_pct", "win", "hard_gain"]]
print(df.sort_values("overall").to_string())
print(f"\n基準: コンセンサス {cons_ov:.4f} / oracle(1本) {or1_ov:.4f} / 初代LGBM 7.9452")
